# 01. Chuẩn bị Dữ liệu

**Mục tiêu:** Load data, tạo target variables, chia train/val/test

**Input:** `features_advanced.csv`
**Output:** Train/Val/Test datasets với targets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*60)
print(" CHUẨN BỊ DỮ LIỆU - DỰ BÁO ĐỘNG ĐẤT ")
print("="*60)

## 1. Load Dữ liệu

In [ ]:
# Load data
df = pd.read_csv('/home/haind/Desktop/earthquake-sequence-mining/haind/features_advanced.csv')

# Hiển thị thông tin cơ bản
print(f"\nShape: {df.shape}")
print(f"Số lượng events: {len(df):,}")
print(f"Số lượng features: {len(df.columns)}")

In [ ]:
# Xem các cột
print("\nDanh sách các cột:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Thông tin dữ liệu
print("\nThông tin dữ liệu:")
df.info()

In [ ]:
# Kiểm tra missing values
print("\nMissing values per column:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)
print(missing_df)

In [ ]:
# Thống kê cơ bản
print("\nThống kê cơ bản:")
print(df[['mag', 'depth', 'time_to_next_days', 'next_mag']].describe())

## 2. Tạo Target Variables

In [ ]:
# Chuyển time sang datetime
df['time'] = pd.to_datetime(df['time'])

# Sắp xếp theo thời gian
df = df.sort_values('time').reset_index(drop=True)
print(f"Đã sắp xếp theo thời gian: {df['time'].min()} đến {df['time'].max()}")

In [ ]:
# Tạo target variables
df['time_to_next'] = df['time'].shift(-1) - df['time']
df['next_mag'] = df['mag'].shift(-1)

# Chuyển time_to_next sang số (ngày)
df['time_to_next_days'] = df['time_to_next'].dt.total_seconds() / 86400

# Xóa row cuối cùng (không có next)
df_original = df.copy()
df = df[:-1].copy()

print(f"\nSau khi tạo targets:")
print(f"Shape: {df.shape}")
print(f"\nTarget - time_to_next_days:")
print(df['time_to_next_days'].describe())

In [ ]:
# Vẽ distribution của targets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time to next
axes[0].hist(df['time_to_next_days'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Ngày')
axes[0].set_ylabel('Số lượng')
axes[0].set_title('Phân phối: Thời gian đến event tiếp theo')
axes[0].axvline(df['time_to_next_days'].median(), color='r', linestyle='--', label=f'Median: {df["time_to_next_days"].median():.1f}')
axes[0].legend()

# Next magnitude
axes[1].hist(df['next_mag'], bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Độ lớn')
axes[1].set_ylabel('Số lượng')
axes[1].set_title('Phân phối: Độ lớn event tiếp theo')
axes[1].axvline(df['next_mag'].median(), color='r', linestyle='--', label=f'Median: {df["next_mag"].median():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Chia Train/Validation/Test

In [ ]:
# Chia theo thời gian (quan trọng cho time-series)
n = len(df)
train_end = int(0.6 * n)
val_end = int(0.8 * n)

train = df.iloc[:train_end].copy()
val = df.iloc[train_end:val_end].copy()
test = df.iloc[val_end:].copy()

print("="*60)
print(" CHIA TRAIN/VALIDATION/TEST ")
print("="*60)
print(f"Train:      {len(train):,} events ({60}%) - {train['time'].min()} đến {train['time'].max()}")
print(f"Validation: {len(val):,} events ({20}%) - {val['time'].min()} đến {val['time'].max()}")
print(f"Test:       {len(test):,} events ({20}%) - {test['time'].min()} đến {test['time'].max()}")

## 4. Chọn Features

In [ ]:
# Features sử dụng cho model
FEATURES = [
    # Basic features
    'latitude', 'longitude', 'depth', 'mag',
    
    # Aftershock features
    'is_aftershock', 'mainshock_id', 'mainshock_mag',
    
    # Fault proximity
    'dist_to_5th_neighbor_km', 'dist_to_10th_neighbor_km',
    'dist_to_20th_neighbor_km', 'seismicity_density_100km',
    
    # Coulomb stress
    'coulomb_stress_proxy',
    
    # Regional features
    'regional_b_value', 'seismic_gap_days', 'regional_max_mag_5yr'
]

TARGET_TIME = 'time_to_next_days'
TARGET_MAG = 'next_mag'

print(f"\nSố lượng features: {len(FEATURES)}")
print(f"\nDanh sách features:")
for i, feat in enumerate(FEATURES, 1):
    print(f"{i:2d}. {feat}")

In [ ]:
# Kiểm tra features có sẵn
missing_features = [f for f in FEATURES if f not in df.columns]
if missing_features:
    print(f"\nWARNING: Các features sau không có trong data:")
    for f in missing_features:
        print(f"  - {f}")
else:
    print(f"\n✓ Tất cả {len(FEATURES)} features đều có sẵn!")

In [ ]:
# Thống kê features trong từng split
print("\nThống kê targets theo split:")
print("-" * 60)

for name, data in [('Train', train), ('Val', val), ('Test', test)]:
    print(f"\n{name}:")
    print(f"  time_to_next_days: mean={data[TARGET_TIME].mean():.2f}, std={data[TARGET_TIME].std():.2f}")
    print(f"  next_mag:         mean={data[TARGET_MAG].mean():.3f}, std={data[TARGET_MAG].std():.3f}")

## 5. Xử lý Missing Values

In [ ]:
# Kiểm tra missing trong FEATURES
print("Missing values trong features (train):")
for feat in FEATURES:
    missing = train[feat].isnull().sum()
    if missing > 0:
        print(f"  {feat}: {missing} ({missing/len(train)*100:.2f}%)")

In [ ]:
# Xử lý missing values
def handle_missing(df):
    df = df.copy()
    
    # Fill regional_b_value với median
    df['regional_b_value'] = df['regional_b_value'].fillna(df['regional_b_value'].median())
    
    # Fill seismic_gap_days với 10 năm
    df['seismic_gap_days'] = df['seismic_gap_days'].fillna(3650)
    
    # Fill mainshock_mag với mag hiện tại (cho non-aftershocks)
    df['mainshock_mag'] = df['mainshock_mag'].fillna(df['mag'])
    
    return df

train = handle_missing(train)
val = handle_missing(val)
test = handle_missing(test)

print("\n✓ Đã xử lý missing values!")

## 6. Lưu Datasets

In [ ]:
# Tạo thư mục output
output_dir = Path('/home/haind/Desktop/earthquake-sequence-mining/haind/data')
output_dir.mkdir(exist_ok=True)

# Lưu datasets
train.to_csv(output_dir / 'train.csv', index=False)
val.to_csv(output_dir / 'val.csv', index=False)
test.to_csv(output_dir / 'test.csv', index=False)

# Lưu feature list
import json
with open(output_dir / 'features.json', 'w') as f:
    json.dump({'FEATURES': FEATURES, 'TARGET_TIME': TARGET_TIME, 'TARGET_MAG': TARGET_MAG}, f, indent=2)

print("="*60)
print(" ĐÃ LƯU DATASETS! ")
print("="*60)
print(f"Train:   {output_dir / 'train.csv'}")
print(f"Val:     {output_dir / 'val.csv'}")
print(f"Test:    {output_dir / 'test.csv'}")
print(f"Config:  {output_dir / 'features.json'}")

## 7. Summary

In [ ]:
print("="*60)
print(" SUMMARY - DATA PREPARATION ")
print("="*60)
print(f"\nTổng số events: {len(df):,}")
print(f"\nSplit:")
print(f"  Train:      {len(train):,} ({len(train)/len(df)*100:.1f}%)")
print(f"  Validation: {len(val):,} ({len(val)/len(df)*100:.1f}%)")
print(f"  Test:       {len(test):,} ({len(test)/len(df)*100:.1f}%)")
print(f"\nFeatures: {len(FEATURES)}")
print(f"\nTargets:")
print(f"  - {TARGET_TIME}: dự đoán thời gian (ngày)")
print(f"  - {TARGET_MAG}: dự đoán độ lớn")
print(f"\nNext steps:")
print(f"  1. Run 02_feature_engineering.ipynb")
print(f"  2. Run 03_time_model.ipynb")
print(f"  3. Run 04_magnitude_model.ipynb")